In [1]:
!jt -t onedork -fs 12 -cellw 95%

import os
import pandas as pd
import folium

from folium.features import DivIcon

In [38]:
pd.set_option('display.max.columns', 50)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 1000)

# Data 탐색

## berth_df: 정계지 위치 정보
### 정계지란, 도선사들이 대기하는 위치

In [2]:
berth_df = pd.read_csv("정계지 위치.csv")

In [5]:
lat, lon = berth_df["위도"].mean(), berth_df["경도"].mean()
map_obj = folium.Map(location=[lat, lon], zoom_start=13)

# CircleMarker 내부에 텍스트 표시
for _, _val in berth_df.iterrows():
    _lat, _lon = _val["위도"], _val["경도"]
    _name = _val["정계지"][:2]
    
    folium.CircleMarker(
        location=[_lat, _lon],
        radius=30,
        color="",
        fill=True,
        fill_color="#ff7800",
        fill_opacity=0.3
    ).add_to(map_obj)
    
    folium.map.Marker(
        [_lat, _lon],
        icon=DivIcon(
            icon_size=(60, 20),  # 아이콘 크기 조정
            icon_anchor=(30, 10),  # 앵커 위치 조정
            html=f'<div style="font-size: 8pt; color : black; text-align: center; font-weight: bold;">{_name}</div>',
        )
    ).add_to(map_obj)

In [6]:
map_obj

주어진 예선데이터를 기반으로 간단한 Toy Problem을 풀어보고자 함

> 데이터 전처리를 진행해보고, 원하는 형태의 df를 제안할 예정

In [ ]:
df = pd.read_excel("예선데이터_인천항.xlsx")

In [40]:
df.columns.tolist()

['기준예선',
 '배정예선',
 '선박명',
 'From',
 '작업 시작지',
 '작업 종료지',
 'To',
 '실제 스케줄 시작 시각',
 '실제 스케줄 종료 시각',
 '작업까지 이동 거리(km)',
 '작업까지 이동 시간(분)',
 '작업중 이동 거리(km)',
 '작업중 이동 시간(분)',
 '도선사',
 '톤수',
 '선종',
 '최종 스케줄 시각',
 '최초 스케줄 시각',
 '최초 작업시작지',
 '최초 작업종료지',
 '최초 예선',
 '최초 도선사',
 'AIS 파일명 target',
 'AIS 파일명 tug']

In [ ]:
col_oi = []

In [44]:
df.head()

,기준예선,배정예선,선박명,From,작업 시작지,작업 종료지,To,실제 스케줄 시작 시각,실제 스케줄 종료 시각,작업까지 이동 거리(km),작업까지 이동 시간(분),작업중 이동 거리(km),작업중 이동 시간(분),도선사,톤수,선종,최종 스케줄 시각,최초 스케줄 시각,최초 작업시작지,최초 작업종료지,최초 예선,최초 도선사,AIS 파일명 target,AIS 파일명 tug
0,I,"성,I",REVERENCE,NaN,NPPS,SNCT2,송도정계지,2024-06-08T07:32:18.000Z,2024-06-08T08:15:18.000Z,8.571,30.000000,4.909,38.000000,KU,9990,풀컨테이너선,2024-06-08T07:00:00.000Z,2024-06-08T07:00:00.000Z,NPPS,SNCT2,성I,KU,쥬피터1호_REVERENCE_2024-06-08_target_1_aisData.csv,쥬피터1호_REVERENCE_2024-06-08_tug_1_aisData.csv
1,M,"F1,M",SEAWAYS MOMENT,NaN,SKI 2,DONG,NaN,2024-06-08T07:10:18.000Z,2024-06-08T07:20:18.000Z,0.626,2.000000,1.859,5.000000,JJ,29293,케미칼 운반선,2024-06-08T07:00:00.000Z,2024-06-08T07:00:00.000Z,SKI 2,DONG,F1M,JJ,뉴마스호_SEAWAYS MOMENT_2024-06-08_target_2_aisData.csv,뉴마스호_SEAWAYS MOMENT_2024-06-08_tug_2_aisData.csv
2,T,"T,S",YURIY TARAPUROV,연안부두정계지,PAL,N53,NaN,2024-06-08T10:51:18.000Z,2024-06-08T11:01:19.000Z,5.673,25.983333,2.561,35.016667,CY,7949,일반화물선,2024-06-08T09:20:00.000Z,2024-06-08T09:00:00.000Z,PAL,N53,TS,CY,대창호_YURIY TARAPUROV_2024-06-08_target_3_aisData.csv,대창호_YURIY TARAPUROV_2024-06-08_tug_3_aisData.csv
3,M,M,BIRYONG,연안부두정계지,PAL,NIPT2,NaN,2024-06-08T10:26:18.000Z,2024-06-08T11:01:18.000Z,5.235,47.000000,3.371,30.000000,KI,14614,국제카페리,2024-06-08T09:50:00.000Z,2024-06-08T09:30:00.000Z,PAL,NIPT2,M,KI,뉴마스호_BIRYONG_2024-06-08_target_4_aisData.csv,뉴마스호_BIRYONG_2024-06-08_tug_4_aisData.csv
4,M,M,XIN XIANG XUE LAN,NaN,PAL,NIPT5,연안부두정계지,2024-06-08T11:20:18.000Z,2024-06-08T11:59:18.000Z,4.024,26.983333,4.567,34.000000,JJ,32729,국제카페리,2024-06-08T10:50:00.000Z,2024-06-08T10:30:00.000Z,PAL,NIPT5,M,NaN,뉴마스호_XIN XIANG XUE LAN_2024-06-08_target_5_aisData.csv,뉴마스호_XIN XIANG XUE LAN_2024-06-08_tug_5_aisData.csv


In [39]:
df

,기준예선,배정예선,선박명,From,작업 시작지,작업 종료지,To,실제 스케줄 시작 시각,실제 스케줄 종료 시각,작업까지 이동 거리(km),작업까지 이동 시간(분),작업중 이동 거리(km),작업중 이동 시간(분),도선사,톤수,선종,최종 스케줄 시각,최초 스케줄 시각,최초 작업시작지,최초 작업종료지,최초 예선,최초 도선사,AIS 파일명 target,AIS 파일명 tug
0,I,"성,I",REVERENCE,NaN,NPPS,SNCT2,송도정계지,2024-06-08T07:32:18.000Z,2024-06-08T08:15:18.000Z,8.571,30.000000,4.909,38.000000,KU,9990,풀컨테이너선,2024-06-08T07:00:00.000Z,2024-06-08T07:00:00.000Z,NPPS,SNCT2,성I,KU,쥬피터1호_REVERENCE_2024-06-08_target_1_aisData.csv,쥬피터1호_REVERENCE_2024-06-08_tug_1_aisData.csv
1,M,"F1,M",SEAWAYS MOMENT,NaN,SKI 2,DONG,NaN,2024-06-08T07:10:18.000Z,2024-06-08T07:20:18.000Z,0.626,2.000000,1.859,5.000000,JJ,29293,케미칼 운반선,2024-06-08T07:00:00.000Z,2024-06-08T07:00:00.000Z,SKI 2,DONG,F1M,JJ,뉴마스호_SEAWAYS MOMENT_2024-06-08_target_2_aisData.csv,뉴마스호_SEAWAYS MOMENT_2024-06-08_tug_2_aisData.csv
2,T,"T,S",YURIY TARAPUROV,연안부두정계지,PAL,N53,NaN,2024-06-08T10:51:18.000Z,2024-06-08T11:01:19.000Z,5.673,25.983333,2.561,35.016667,CY,7949,일반화물선,2024-06-08T09:20:00.000Z,2024-06-08T09:00:00.000Z,PAL,N53,TS,CY,대창호_YURIY TARAPUROV_2024-06-08_target_3_aisData.csv,대창호_YURIY TARAPUROV_2024-06-08_tug_3_aisData.csv
3,M,M,BIRYONG,연안부두정계지,PAL,NIPT2,NaN,2024-06-08T10:26:18.000Z,2024-06-08T11:01:18.000Z,5.235,47.000000,3.371,30.000000,KI,14614,국제카페리,2024-06-08T09:50:00.000Z,2024-06-08T09:30:00.000Z,PAL,NIPT2,M,KI,뉴마스호_BIRYONG_2024-06-08_target_4_aisData.csv,뉴마스호_BIRYONG_2024-06-08_tug_4_aisData.csv
4,M,M,XIN XIANG XUE LAN,NaN,PAL,NIPT5,연안부두정계지,2024-06-08T11:20:18.000Z,2024-06-08T11:59:18.000Z,4.024,26.983333,4.567,34.000000,JJ,32729,국제카페리,2024-06-08T10:50:00.000Z,2024-06-08T10:30:00.000Z,PAL,NIPT5,M,NaN,뉴마스호_XIN XIANG XUE LAN_2024-06-08_target_5_aisData.csv,뉴마스호_XIN XIANG XUE LAN_2024-06-08_tug_5_aisData.csv
5,N,"N,O",SEONGHO PIOCE,연안부두정계지,PAL,SKI 2,북항임시정계지,2024-06-08T15:02:19.000Z,2024-06-08T15:43:19.000Z,7.567,30.000000,2.935,36.000000,KK,3698,석유제품 운반선,2024-06-08T14:10:00.000Z,2024-06-08T14:00:00.000Z,PAL,SKI 2,NO,NaN,뉴엔젤호_SEONGHO PIOCE_2024-06-08_target_6_aisData.csv,뉴엔젤호_SEONGHO PIOCE_2024-06-08_tug_6_aisData.csv
6,T,"R,T",QUEZON BRIDGE,연안부두정계지,PAL,ICT1,연안부두정계지,2024-06-08T15:03:19.000Z,2024-06-08T15:44:19.000Z,6.435,36.000000,3.391,36.000000,LY,17211,풀컨테이너선,2024-06-08T14:20:00.000Z,2024-06-08T14:30:00.000Z,PAL,ICT1,RT,NaN,대창호_QUEZON BRIDGE_2024-06-08_target_7_aisData.csv,대창호_QUEZON BRIDGE_2024-06-08_tug_7_aisData.csv
7,I,"K,I",KMTC LAEM CHABANG,송도정계지,NPPS,HJIT3,송도정계지,2024-06-09T06:53:19.000Z,2024-06-09T07:43:19.000Z,2.364,17.000000,6.017,45.000000,RW,18318,풀컨테이너선,2024-06-09T06:30:00.000Z,2024-06-09T06:30:00.000Z,NPPS,HJIT3,KI,RW,쥬피터1호_KMTC LAEM CHABANG_2024-06-09_target_8_aisData.csv,쥬피터1호_KMTC LAEM CHABANG_2024-06-09_tug_8_aisData.csv
8,명,"L,F2,B,비,명",GOOD LUCK,내항정계지,LEN,43,내항정계지,2024-06-09T08:13:19.000Z,2024-06-09T09:10:19.000Z,1.213,8.000000,3.173,52.000000,JH,41074,산물선(벌크선),2024-06-09T06:30:00.000Z,2024-06-09T06:30:00.000Z,LEN,43,LF2B비명,JH,대명호_GOOD LUCK_2024-06-09_target_9_aisData.csv,대명호_GOOD LUCK_2024-06-09_tug_9_aisData.csv
9,N,"N,Q",WOO CHOON,북항임시정계지,GSD,PAL,NaN,2024-06-09T09:45:19.000Z,2024-06-09T10:07:19.000Z,4.850,24.000000,1.422,17.000000,NaN,3969,석유제품 운반선,2024-06-09T10:00:00.000Z,2024-06-09T10:00:00.000Z,GSD,PAL,NQ,NaN,뉴엔젤호_WOO CHOON_2024-06-09_target_10_aisData.csv,뉴엔젤호_WOO CHOON_2024-06-09_tug_10_aisData.csv


In [32]:
df["From"].unique()

array([nan, '연안부두정계지', '송도정계지', '내항정계지', '북항임시정계지'], dtype=object)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336 entries, 0 to 335
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   기준예선            336 non-null    object 
 1   배정예선            336 non-null    object 
 2   선박명             336 non-null    object 
 3   From            217 non-null    object 
 4   작업 시작지          336 non-null    object 
 5   작업 종료지          336 non-null    object 
 6   To              190 non-null    object 
 7   실제 스케줄 시작 시각    336 non-null    object 
 8   실제 스케줄 종료 시각    336 non-null    object 
 9   작업까지 이동 거리(km)  336 non-null    float64
 10  작업까지 이동 시간(분)   336 non-null    float64
 11  작업중 이동 거리(km)   336 non-null    float64
 12  작업중 이동 시간(분)    336 non-null    float64
 13  도선사             252 non-null    object 
 14  톤수              336 non-null    int64  
 15  선종              336 non-null    object 
 16  최종 스케줄 시각       336 non-null    object 
 17  최초 스케줄 시각       336 non-null    obj

In [28]:
df.dropna().info()

<class 'pandas.core.frame.DataFrame'>
Index: 48 entries, 7 to 317
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   기준예선            48 non-null     object 
 1   배정예선            48 non-null     object 
 2   선박명             48 non-null     object 
 3   From            48 non-null     object 
 4   작업 시작지          48 non-null     object 
 5   작업 종료지          48 non-null     object 
 6   To              48 non-null     object 
 7   실제 스케줄 시작 시각    48 non-null     object 
 8   실제 스케줄 종료 시각    48 non-null     object 
 9   작업까지 이동 거리(km)  48 non-null     float64
 10  작업까지 이동 시간(분)   48 non-null     float64
 11  작업중 이동 거리(km)   48 non-null     float64
 12  작업중 이동 시간(분)    48 non-null     float64
 13  도선사             48 non-null     object 
 14  톤수              48 non-null     int64  
 15  선종              48 non-null     object 
 16  최종 스케줄 시각       48 non-null     object 
 17  최초 스케줄 시각       48 non-null     object 
 

In [24]:
type(df.From.iloc[0])

float

In [10]:
df = pd.read_excel("예선데이터_인천항.xlsx")

In [12]:
df.columns

Index(['기준예선', '배정예선', '선박명', 'From', '작업 시작지', '작업 종료지', 'To', '실제 스케줄 시작 시각',
       '실제 스케줄 종료 시각', '작업까지 이동 거리(km)', '작업까지 이동 시간(분)', '작업중 이동 거리(km)',
       '작업중 이동 시간(분)', '도선사', '톤수', '선종', '최종 스케줄 시각', '최초 스케줄 시각', '최초 작업시작지',
       '최초 작업종료지', '최초 예선', '최초 도선사', 'AIS 파일명 target', 'AIS 파일명 tug'],
      dtype='object')

In [ ]:
pd.read_csv("선석 코드.csv")

In [ ]:
pd.read_csv("선석 코드.csv")["선석코드"].unique()

In [ ]:
df = pd.read_csv("2024-06_FristAllSchData.csv")

In [ ]:
df["도선사"].unique()

In [ ]:
pd.read_csv("2024-06_FristAllSchData.csv")

In [ ]:
pd.read_csv("2024-06_SchData.csv")